In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Introducere în primitive

<Accordion>
<AccordionItem title="Versiuni de pachete">

Codul de pe această pagină a fost dezvoltat folosind cerințele de mai jos.
Îți recomandăm să folosești aceste versiuni sau unele mai noi.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</AccordionItem>
</Accordion>
<span id="qpu-access-patterns"></span>
## De ce a introdus Qiskit primitive?
Similar cu primele zile ale calculatoarelor clasice, când dezvoltatorii trebuiau să manipuleze direct registrele CPU, interfața timpurie cu QPU-urile returna pur și simplu datele brute de la electronica de control.
Aceasta nu era o problemă majoră atunci când QPU-urile se aflau în laboratoare și permiteau accesul direct doar cercetătorilor.
Recunoscând că majoritatea dezvoltatorilor nu ar trebui și nu ar trebui să fie familiari cu distilarea unor astfel de date brute în 0 și 1, Qiskit a introdus `backend.run`, o primă abstractizare pentru accesarea QPU-urilor în cloud. Aceasta le-a permis dezvoltatorilor
să opereze într-un format de date familiar și să se concentreze pe imaginea de ansamblu.

Pe măsură ce accesul la QPU-uri a devenit mai răspândit, iar mai mulți algoritmi cuantici au fost dezvoltați,
a apărut din nou nevoia unei abstractizări de nivel mai înalt. Ca răspuns, Qiskit a introdus
interfața de primitive, optimizată pentru două sarcini de bază în dezvoltarea algoritmilor cuantici:
estimarea valorii de așteptare (`Estimator`) și eșantionarea circuitelor (`Sampler`). Obiectivul este din nou
să îi ajute pe dezvoltatori să se concentreze mai mult pe inovație și mai puțin pe conversia datelor. Interfața de primitive înlocuiește interfața `backend.run`, deoarece `Sampler` oferă același acces direct la hardware care era oferit de `backend.run`.

## Ce este o primitivă?
Sistemele de calcul sunt construite pe mai multe niveluri de abstractizare. Abstractizările îți permit să te concentrezi pe
un anumit nivel de detaliu relevant pentru sarcina în curs. Cu cât ești mai aproape de hardware,
cu atât ai nevoie de un nivel mai scăzut de abstractizare (de exemplu, ar putea fi necesar să muți sau să manipulezi date la nivelul instrucțiunilor CPU). Cu cât sarcina pe care vrei să o efectuezi este mai complexă,
cu atât abstractizările vor fi de nivel mai înalt (de exemplu, ai putea folosi o bibliotecă de programare pentru a efectua
calcule algebrice).

În acest context, o *primitivă* este cea mai mică instrucțiune de procesare, cel mai simplu bloc de construcție din care
cineva poate crea ceva util pentru un anumit nivel de abstractizare.

Progresele recente în domeniul calculului cuantic au crescut nevoia de a lucra la niveluri mai înalte de abstractizare.
Pe măsură ce domeniul se îndreaptă spre unități de procesare cuantică (QPU) mai mari și fluxuri de lucru mai complexe, accentul se mută de la interacțiunea cu semnalele individuale ale qubiților la vizualizarea dispozitivelor cuantice ca sisteme care efectuează sarcini necesare.

Cele mai frecvente două sarcini pentru calculatoarele cuantice sunt eșantionarea stărilor cuantice și calcularea valorilor de așteptare.
Aceste sarcini au motivat designul primitivelor Qiskit: **Estimator** și **Sampler**.

- Estimator calculează valorile de așteptare ale observabilelor față de stările pregătite de Circuit-uri cuantice.
- Sampler eșantionează registrul de ieșire din execuția Circuit-ului cuantic.

Pe scurt, modelul de calcul introdus de primitivele Qiskit mută programarea cuantică cu un pas mai aproape
de unde se află astăzi programarea clasică, unde accentul este mai puțin pe detaliile hardware și mai mult pe rezultatele
pe care încerci să le obții.

## Definiția și implementările primitivelor
Există două tipuri de primitive Qiskit: clasele de bază și implementările lor. Primitivele Estimator și Sampler sunt definite prin clase de bază primitive open-source care se află în Qiskit SDK (în modulul [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives)). Furnizorii (cum ar fi Qiskit Runtime) pot folosi aceste clase de bază pentru a-și deriva propriile implementări de Sampler și Estimator. Majoritatea utilizatorilor vor interacționa cu implementările furnizorilor, nu cu primitivele de bază.

### Clase de bază
Primitivele `Base` sunt clase abstracte care definesc o interfață comună pentru implementarea primitivelor. Toate celelalte clase din modulul [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) moștenesc din aceste clase de bază. Dezvoltatorii ar trebui să le folosească dacă sunt interesați să creeze propriul model de execuție bazat pe primitive pentru un anumit furnizor. Aceste clase ar putea fi, de asemenea, utile pentru cei care doresc să efectueze procesări foarte personalizate și consideră că implementările existente de primitive sunt prea simple pentru nevoile lor. Utilizatorii generali nu vor folosi direct clasele de bază.

[`BaseEstimatorV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV1) și [`BaseSamplerV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV1) - Deși primitivele V1 sunt încă utilizabile, aceste ghiduri se concentrează pe primitivele V2, deoarece acestea sunt cele mai recente și sunt folosite mai frecvent.

[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) și [`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) - Primitivele de referință Qiskit urmează aceste specificații de interfață.

<span id="implementations"></span>
### Implementări
Toate primitivele sunt create din clasele de bază; prin urmare, au aceeași structură și utilizare generală. De exemplu, formatul intrării pentru toate primitivele Estimator este același. Cu toate acestea, există diferențe în implementări care le fac unice.

Acestea sunt implementări ale claselor de bază primitive:

- [Primitivele Qiskit Runtime](/guides/qiskit-runtime-primitives), [`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) și [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2), oferă o implementare mai sofisticată (de exemplu, incluzând atenuarea erorilor) ca serviciu bazat pe cloud. Această implementare a primitivelor de bază este folosită pentru a accesa hardware-ul IBM Quantum&reg;.

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator) și [`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) - Implementări de referință ale primitivelor care folosesc simulatorul integrat în Qiskit. Sunt construite cu modulul Qiskit [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information), producând rezultate bazate pe simulări ideale de tip statevector. Sunt accesate prin Qiskit. Consultă [Simulare exactă cu primitivele Qiskit](/guides/simulate-with-qiskit-sdk-primitives) pentru detalii de utilizare.

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) și [`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) - Poți folosi aceste clase pentru a „împacheta" orice resursă de calcul cuantic într-o primitivă. Aceasta îți permite să scrii cod în stil de primitivă pentru furnizorii care nu au încă o interfață bazată pe primitive. Aceste clase pot fi folosite la fel ca Sampler și Estimator obișnuite, cu excepția că trebuie inițializate cu un argument suplimentar `backend` pentru a selecta pe ce calculator cuantic să ruleze. Sunt accesate folosind Qiskit. Consultă ghidul [primitive backend](/guides/get-started-with-backend-primitives) pentru mai multe informații.

## Opțiuni
Poți transmite opțiuni primitivelor pentru a le personaliza în funcție de nevoile tale. Deși interfața metodei `run()` a primitivelor este comună tuturor implementărilor, opțiunile lor nu sunt. Consultă referințele API pentru o implementare specifică de primitivă pentru a afla ce opțiuni suportă.

De exemplu, consultă subiectele [Opțiuni Estimator](/guides/estimator-options) și [Opțiuni Sampler](/guides/sampler-options) pentru a afla despre opțiunile primitivelor Qiskit Runtime, sau vezi [referințele API Qiskit Aer](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) pentru opțiunile primitivelor Qiskit Aer.

## Beneficiile primitivelor Qiskit
Cu primitive, utilizatorii Qiskit pot scrie cod cuantic pentru un QPU specific fără a trebui să gestioneze explicit
fiecare detaliu. De asemenea, datorită nivelului suplimentar de abstractizare, este posibil să poți accesa mai ușor
capabilitățile hardware avansate ale unui anumit furnizor. De exemplu, cu primitivele Qiskit Runtime,
poți profita de cele mai recente progrese în atenuarea și suprimarea erorilor prin comutarea unor opțiuni precum [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level) al primitivei, în loc să construiești propria implementare a acestor tehnici.

Pentru furnizorii de hardware, implementarea nativă a primitivelor înseamnă că poți oferi utilizatorilor tăi un mod mai „gata de utilizare"
de a accesa funcțiile hardware, cum ar fi tehnicile avansate de post-procesare. Astfel, este mai ușor pentru utilizatorii tăi să beneficieze de cele mai bune capabilități ale hardware-ului tău.

## Pași următori
> **Tip:** - Înțelege [intrarea și ieșirea primitivelor](/guides/primitive-input-output).
> - Consultă [exemple detaliate](/guides/simulate-with-qiskit-sdk-primitives).
> - Exersează cu primitivele parcurgând [lecția despre funcții de cost](/learning/courses/variational-algorithm-design/cost-functions) în IBM Quantum Learning.
> - Consultă [Creează un furnizor](/guides/create-a-provider) pentru a afla cum să implementezi propriile primitive Sampler și Estimator.
> - Consultă [referințele API](https://docs.quantum.ibm.com/api/qiskit/primitives).
> - Citește [Migrează la primitivele V2](/guides/v2-primitives).
> - Află despre [primitivele Qiskit Runtime](/guides/qiskit-runtime-primitives), care sunt folosite pentru rularea circuitelor pe QPU-uri IBM.